In [1]:
import bigframes.pandas as bpd

bpd.options.bigquery.ordering_mode = "partial"

In [12]:
df = bpd.read_gbq_table("bigquery-public-data.noaa_lightning.lightning_strikes")

In [17]:
df.shape

(111904021938, 5)

In [13]:
df.dtypes

date                 timestamp[us, tz=UTC][pyarrow]
number_of_strikes                             Int64
center_point_geom                          geometry
source_url                                   string
etl_timestamp        timestamp[us, tz=UTC][pyarrow]
dtype: object

In [15]:
df['date'].max()

Timestamp('2025-12-31 00:00:00+0000', tz='UTC')

In [16]:
import bigframes.bigquery as bbq
from shapely.geometry import Point

minneapolis_pt = Point(-93.26384, 44.97997)

near_minneapolis = bbq.st_distance(df['center_point_geom'], minneapolis_pt) < 100_000
near_minneapolis.sum()

np.int64(287369248)

In [19]:
strikes = df[near_minneapolis].groupby("date").agg({"number_of_strikes": "sum"})

In [20]:
strikes_pd = strikes.to_pandas()
strikes_pd

,number_of_strikes
date,
1990-01-23 00:00:00+00:00,1129
1990-03-11 00:00:00+00:00,19193
1990-03-13 00:00:00+00:00,672884
1990-03-14 00:00:00+00:00,305959
1990-03-15 00:00:00+00:00,5645
...,...
2025-08-17 00:00:00+00:00,170821
2025-08-18 00:00:00+00:00,123809
2025-08-22 00:00:00+00:00,805


In [21]:
strikes_pd.to_parquet("lightning.parquet")